#  Transfer Learning with ResNet50: Oxford Flowers 102

This notebook demonstrates feature extraction and fine-tuning with a pretrained **ResNet50** model.

## Table of Contents
1. Transfer Learning Overview
2. Dataset Loading
3. Preprocessing
4. Feature Extraction
5. Fine-Tuning
6. Learning Curve Comparison
7. Prediction Visualization
8. Grad-CAM
9. Key Learnings, Interview Questions, Common Mistakes, Further Reading


## 1. Transfer Learning Overview

Transfer learning reuses visual features learned from a large dataset such as ImageNet. Instead of training all layers from scratch, we first freeze the pretrained backbone and train a small classification head.

```mermaid
flowchart LR
    A[Flower image] --> B[ImageNet ResNet50 backbone]
    B --> C[Global pooling]
    C --> D[New classification head]
    D --> E[102 flower species]
```

**Feature extraction:** freeze the backbone and train only new top layers.

**Fine-tuning:** unfreeze selected deeper layers and adapt them gently with a small learning rate.


In [ ]:
# Reproducibility and common imports
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
try:
    import tensorflow as tf
    import tensorflow_datasets as tfds
except ImportError:
    %pip install -q tensorflow tensorflow-datasets
    import tensorflow as tf
    import tensorflow_datasets as tfds

tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


## 2. Load Oxford Flowers 102

TensorFlow Datasets provides a reproducible split and class labels for this dataset.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

(ds_train, ds_val, ds_test), info = tfds.load(
    "oxford_flowers102",
    split=["train", "validation", "test"],
    as_supervised=True,
    with_info=True,
)
NUM_CLASSES = info.features["label"].num_classes
print(info)
print("Classes:", NUM_CLASSES)


## 3. Preprocessing

ResNet50 expects inputs preprocessed with the same convention used during ImageNet training.


In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED)
    return ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = prepare(ds_train, shuffle=True)
val_ds = prepare(ds_val)
test_ds = prepare(ds_test)


In [ ]:
# Display original samples before ResNet preprocessing.
plt.figure(figsize=(12, 8))
for i, (image, label) in enumerate(tfds.as_numpy(ds_train.take(9))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(f"Class {label}")
    plt.axis("off")
plt.tight_layout()


## 4. Feature Extraction: Frozen ResNet50 Backbone


In [ ]:
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs, name="flowers_resnet50_feature_extractor")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
feature_history = model.fit(train_ds, validation_data=val_ds, epochs=8)


## 5. Fine-Tuning

Only the last ResNet block is unfrozen. A small learning rate prevents destroying useful pretrained features.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
fine_history = model.fit(train_ds, validation_data=val_ds, epochs=8)


## 6. Compare Before and After Fine-Tuning


In [ ]:
def merge_histories(*histories):
    merged = {}
    for history in histories:
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(values)
    return merged

history_all = merge_histories(feature_history, fine_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(history_all[metric], label=f"train_{metric}", linewidth=2)
    ax.plot(history_all[f"val_{metric}"], label=f"val_{metric}", linewidth=2)
    ax.axvline(len(feature_history.history[metric]) - 1, color="black", linestyle="--", label="fine-tuning starts")
    ax.set_title(metric.title())
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()

test_loss, test_acc = model.evaluate(test_ds)
print(f"Fine-tuned test accuracy: {test_acc:.3f}")


## 7. Display Predictions


In [ ]:
def show_flower_predictions(dataset, model, n=9):
    images, labels = next(iter(dataset))
    probs = model.predict(images)
    preds = probs.argmax(axis=1)
    display_images = (images.numpy() - images.numpy().min()) / (images.numpy().max() - images.numpy().min())
    plt.figure(figsize=(12, 8))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(display_images[i])
        color = "green" if preds[i] == labels[i] else "red"
        plt.title(f"Pred: {preds[i]} | True: {int(labels[i])}", color=color)
        plt.axis("off")
    plt.tight_layout()

show_flower_predictions(test_ds, model)


## 8. Grad-CAM Visualization

Grad-CAM highlights spatial regions that strongly influenced the predicted class. This is useful for model debugging and portfolio storytelling.


In [ ]:
def make_gradcam_heatmap(img_array, model, backbone_name="resnet50", last_conv_layer_name="conv5_block3_out"):
    backbone = model.get_layer(backbone_name)
    last_conv_layer = backbone.get_layer(last_conv_layer_name)
    last_conv_layer_model = tf.keras.Model(backbone.inputs, last_conv_layer.output)

    classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
    x = classifier_input
    start_collecting = False
    for layer in model.layers:
        if layer.name == backbone_name:
            start_collecting = True
            continue
        if start_collecting:
            x = layer(x)
    classifier_model = tf.keras.Model(classifier_input, x)

    with tf.GradientTape() as tape:
        conv_outputs = last_conv_layer_model(img_array)
        tape.watch(conv_outputs)
        predictions = classifier_model(conv_outputs)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

images, labels = next(iter(test_ds))
plt.figure(figsize=(12, 8))
for i in range(6):
    heatmap = make_gradcam_heatmap(images[i:i+1], model)
    display_image = (images[i].numpy() - images[i].numpy().min()) / (images[i].numpy().max() - images[i].numpy().min())
    resized_heatmap = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy().squeeze()
    ax = plt.subplot(2, 3, i + 1)
    ax.imshow(display_image)
    ax.imshow(resized_heatmap, cmap="jet", alpha=0.35)
    ax.set_title(f"True class: {int(labels[i])}")
    ax.axis("off")
plt.tight_layout()


## Key Learnings

- Pretrained CNNs are strong feature extractors.
- Fine-tuning should use a smaller learning rate than head training.
- Grad-CAM helps inspect whether the model looks at meaningful image regions.

## Interview Questions

1. Why freeze a pretrained backbone at first?
2. What can go wrong if the fine-tuning learning rate is too high?
3. How does global average pooling reduce parameters?
4. What does Grad-CAM explain and what does it not explain?

## Common Mistakes

- Forgetting the model-specific preprocessing function.
- Unfreezing too many layers too early.
- Comparing models without the same validation split.
- Treating Grad-CAM as a perfect causal explanation.

## Further Reading

- Keras transfer learning guide: https://keras.io/guides/transfer_learning/
- ResNet paper: https://arxiv.org/abs/1512.03385
- Grad-CAM paper: https://arxiv.org/abs/1610.02391
